# Module 03 — AI Agents
## Lesson 5 — Planning

Planning is useful when a goal has several dependencies, expensive actions, ordering constraints, or a need for review before execution.

> **A plan is an explicit, revisable description of intended work — not permission to execute it.**

This lesson uses a concise external task list. It does not ask the model to expose private chain-of-thought.


### Reactive loop vs plan-first agent

A reactive agent chooses one action at a time. A plan-first agent first creates a small inspectable plan, then executes and revises it when observations invalidate assumptions. Planning costs extra tokens, latency, state, and orchestration, so use it only when it adds value.


### Scenario

> **Plan a half-day outdoors in Melbourne tomorrow. Check whether the weather is suitable and make sure the suggested time is before sunset.**

We reuse weather and sun-time capabilities from earlier lessons so the new concept is planning, not another API integration.


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from datetime import date, timedelta
from functools import lru_cache
from typing import Any, Literal

import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
model = os.getenv("OPENAI_MODEL", "gpt-5.6")


### 1. Make the plan application state

The host tracks step status deterministically instead of asking the model to remember what has completed.


In [ ]:
StepStatus = Literal["pending", "completed", "skipped", "blocked"]

@dataclass
class PlanStep:
    id: int
    action: str
    purpose: str
    status: StepStatus = "pending"
    result: dict[str, Any] | None = None

@dataclass
class PlanState:
    goal: str
    steps: list[PlanStep] = field(default_factory=list)
    revision: int = 0

    def pending(self) -> list[PlanStep]:
        return [step for step in self.steps if step.status == "pending"]


### 2. Ask the model for a concise external plan

The planner is constrained to three high-level actions. The output is an inspectable task list, not hidden reasoning.


In [ ]:
PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "minItems": 1,
            "maxItems": 5,
            "items": {
                "type": "object",
                "properties": {
                    "action": {
                        "type": "string",
                        "enum": ["get_weather", "get_sun_times", "synthesize"],
                    },
                    "purpose": {"type": "string"},
                },
                "required": ["action", "purpose"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["steps"],
    "additionalProperties": False,
}

def create_plan(goal: str) -> PlanState:
    response = client.responses.create(
        model=model,
        instructions=(
            "Create a concise external execution plan. "
            "Do not provide hidden reasoning. Use only allowed actions. "
            "Use the minimum number of steps needed."
        ),
        input=goal,
        text={"format": {"type": "json_schema", "name": "plan", "schema": PLAN_SCHEMA, "strict": True}},
    )
    payload = json.loads(response.output_text)
    steps = [
        PlanStep(i + 1, item["action"], item["purpose"])
        for i, item in enumerate(payload["steps"])
    ]
    return PlanState(goal=goal, steps=steps)


### 3. Validate the plan

A model-generated plan is untrusted input. The host still decides which actions exist and what shape a valid plan has.


In [ ]:
ALLOWED_ACTIONS = {"get_weather", "get_sun_times", "synthesize"}

def validate_plan(plan: PlanState) -> None:
    if not plan.steps:
        raise ValueError("Plan must contain at least one step")
    if len(plan.steps) > 5:
        raise ValueError("Plan is too large")
    if any(step.action not in ALLOWED_ACTIONS for step in plan.steps):
        raise ValueError("Plan contains an unsupported action")
    if plan.steps[-1].action != "synthesize":
        raise ValueError("Plan must end with synthesis")


### 4. Reuse the live Open-Meteo helpers

These are ordinary trusted application functions. Planning changes orchestration, not the tool implementation.


In [ ]:
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

@lru_cache(maxsize=32)
def resolve_city(city: str) -> dict[str, Any]:
    response = requests.get(GEOCODING_URL, params={"name": city, "count": 1, "language": "en", "format": "json"}, timeout=10)
    response.raise_for_status()
    results = response.json().get("results", [])
    if not results:
        raise ValueError(f"Could not resolve city: {city}")
    return results[0]

def fetch_daily(city: str, target_date: str, daily: str) -> dict[str, Any]:
    place = resolve_city(city)
    response = requests.get(FORECAST_URL, params={
        "latitude": place["latitude"], "longitude": place["longitude"],
        "daily": daily, "timezone": "auto",
        "start_date": target_date, "end_date": target_date,
    }, timeout=10)
    response.raise_for_status()
    return response.json()["daily"]

def get_weather(city: str, target_date: str) -> dict[str, Any]:
    daily = fetch_daily(city, target_date, "precipitation_probability_max,temperature_2m_max,temperature_2m_min")
    return {
        "date": daily["time"][0],
        "precipitation_probability_max_percent": daily["precipitation_probability_max"][0],
        "temperature_max_c": daily["temperature_2m_max"][0],
        "temperature_min_c": daily["temperature_2m_min"][0],
    }

def get_sun_times(city: str, target_date: str) -> dict[str, Any]:
    daily = fetch_daily(city, target_date, "sunrise,sunset")
    return {"date": daily["time"][0], "sunrise": daily["sunrise"][0], "sunset": daily["sunset"][0]}


### 5. Create and inspect the plan

Before executing anything, inspect the task list. This is one reason explicit planning can be useful: intended work becomes visible.


In [ ]:
tomorrow = (date.today() + timedelta(days=1)).isoformat()
goal = "Plan a half-day outdoors in Melbourne tomorrow. Check whether the weather is suitable and make sure the suggested time is before sunset."
plan = create_plan(goal)
validate_plan(plan)
[(step.id, step.action, step.purpose, step.status) for step in plan.steps]


### 6. Execute planned evidence-gathering steps

The host executes only allowed tool steps and marks them complete. `synthesize` is handled separately because it is not an external side effect.


In [ ]:
observations: list[dict[str, Any]] = []

for step in plan.steps:
    if step.action == "synthesize":
        break
    if step.action == "get_weather":
        result = get_weather("Melbourne", tomorrow)
    elif step.action == "get_sun_times":
        result = get_sun_times("Melbourne", tomorrow)
    else:
        raise ValueError(step.action)

    step.result = result
    step.status = "completed"
    observations.append(result)

[(step.action, step.status) for step in plan.steps]


### 7. Decide whether the plan is still valid

A plan is a hypothesis. If severe rain invalidates the outdoor assumption, remaining outdoor work should be revised or skipped instead of executed mechanically.


In [ ]:
def should_replan(observations: list[dict[str, Any]]) -> bool:
    for observation in observations:
        rain = observation.get("precipitation_probability_max_percent")
        if rain is not None and rain >= 80:
            return True
    return False

should_replan(observations)


### 8. Final synthesis

The final model call receives the goal plus selected observations. The plan helped coordinate work, but the answer should be grounded in actual evidence rather than the original plan.


In [ ]:
final_response = client.responses.create(
    model=model,
    instructions=(
        "You are a concise travel assistant. Use the supplied observations as evidence. "
        "Do not invent weather or daylight facts."
    ),
    input=(
        f"Goal: {goal}\n\n"
        f"Observations: {json.dumps(observations, indent=2)}"
    ),
)
print(final_response.output_text)


## What to take away

- Planning is useful when dependencies and coordination matter.
- The plan is working state, not durable memory.
- A model-generated plan must be validated before execution.
- Plans should be concise external task lists, not hidden chain-of-thought.
- New observations can invalidate a plan. Replan only when necessary and bound revisions.
- Simple tasks are often better served by a reactive loop or deterministic workflow.

Next: **Lesson 6 — Error Handling and Retries**.
